In [1]:
import argparse
import csv
import glob
import re
import sys
from collections import defaultdict
from pathlib import Path
import pandas as pd
from dataclasses import dataclass
from IPython.display import display

def find_repo_root(start):
    for path in [start, *start.parents]:
        if (path / 'tools' / 'fsdb_cli').exists():
            return path
    raise RuntimeError('Repository root not found')

REPO_ROOT = find_repo_root(Path.cwd())
LATENCY_DIR = REPO_ROOT / 'analysis_workspace' / 'latency'
TOOLS_DIR = REPO_ROOT / 'tools'
for path in (LATENCY_DIR, TOOLS_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import fsdb_cli as fsdb
import cycle_util

In [2]:
@dataclass
class Workload:
    name: str = 'kernel'
    kernel_type: str = 'generic'
    m: int = 0
    n: int = 0
    k: int = 0

    @property
    def is_gemm(self):
        return self.kernel_type.lower() in {'gemm', 'gemv', 'fpint_gemm', 'fpint_gemv'}


@dataclass
class RunResult:
    workload: Workload
    fsdb_path: Path
    simv_log: Path
    elf_path: Path
    phases: pd.DataFrame
    sync_wait: pd.DataFrame | None = None
    mpm_accel: pd.DataFrame | None = None
    util_summary: pd.DataFrame | None = None
    fire_intervals: pd.DataFrame | None = None
    hbm_access_intervals: pd.DataFrame | None = None
    hbm_read_latency: pd.DataFrame | None = None


In [3]:
def _path_or_default(value, default):
    if value is None or value == '':
        return Path(default)
    return Path(value)


def _default_simv_log(fsdb_path):
    return Path(fsdb_path).with_name('simv.log')


def _mpm_value(mpm_accel, section, metric, default=0.0):
    if mpm_accel is None or mpm_accel.empty:
        return default
    value = mpm_accel.loc[(mpm_accel['section'] == section) & (mpm_accel['metric'] == metric), 'value']
    return default if value.empty else value.iloc[0]


def _phase_value(phases, phase, default=0.0):
    if phases is None or phases.empty:
        return default
    value = phases.loc[phases['phase'] == phase, 'cycles']
    return default if value.empty else value.iloc[0]


def _safe_pct(num, den):
    return 0.0 if den == 0 else 100.0 * num / den


def build_util_summary(phases, mpm_accel):
    """Build a compact utilization table from MPM-derived counters."""
    if mpm_accel is None or mpm_accel.empty:
        return pd.DataFrame()

    rows = []

    def add(section, metric, value, unit='pct'):
        rows.append({'section': section, 'metric': metric, 'value': value, 'unit': unit})

    kernel_busy = _phase_value(phases, 'kernel_busy')
    user_body = _phase_value(phases, 'user_kernel_body')
    gemm_total = _mpm_value(mpm_accel, 'mxu', 'gemm_total_cycles')
    gemm_compute = _mpm_value(mpm_accel, 'mxu', 'gemm_compute_cycles')
    gemm_stall = _mpm_value(mpm_accel, 'mxu', 'gemm_stall_cycles')

    add('mxu', 'active_pct_kernel_busy', _safe_pct(gemm_total, kernel_busy))
    add('mxu', 'active_pct_user_kernel_body', _safe_pct(gemm_total, user_body))
    add('mxu', 'compute_pct_gemm_total', _safe_pct(gemm_compute, gemm_total))
    add('mxu', 'stall_pct_gemm_total', _safe_pct(gemm_stall, gemm_total))
    add('mxu', 'flops_per_gemm_cycle', _mpm_value(mpm_accel, 'mxu', 'achieved_flops_per_cycle_total'), 'flop/cycle')
    add('mxu', 'overlap_dma_mxu_pct_dma_active', _mpm_value(mpm_accel, 'mxu', 'overlap_dma_mxu_pct_dma_active'))

    for port in ('input', 'weight', 'psum', 'output'):
        add('mxu_port', f'{port}_util_pct_total', _mpm_value(mpm_accel, 'mxu', f'{port}_util_pct_total'))
        add('mxu_port', f'{port}_util_pct_compute', _mpm_value(mpm_accel, 'mxu', f'{port}_util_pct_compute'))
        add('mxu_port', f'{port}_stall_pct_activity', _mpm_value(mpm_accel, 'mxu', f'{port}_stall_pct_activity'))

    for section in ('hbm_dma', 'ldma_input', 'ldma_weight', 'ldma_sz', 'ldma_output'):
        active = _mpm_value(mpm_accel, section, 'active_cycles')
        add(section, 'active_pct_gemm_total', _safe_pct(active, gemm_total))
        add(section, 'active_pct_user_kernel_body', _safe_pct(active, user_body))
        add(section, 'util_pct_busy', _mpm_value(mpm_accel, section, 'util_pct_busy'))
        add(section, 'util_pct_total', _mpm_value(mpm_accel, section, 'util_pct_total'))
        add(section, 'bandwidth_bytes_per_active_cycle', _mpm_value(mpm_accel, section, 'bandwidth_bytes_per_active_cycle'), 'B/cycle')

    return pd.DataFrame(rows)


def run(
    fsdb_path=None,
    workload=None,
    *,
    simv_log=None,
    elf_path=None,
    paths=None,
    include_gemm=None,
    show=True,
    strict=False,
    include_intervals=None,
    interval_groups=None,
    interval_specs=None,
    include_hbm_access=None,
    hbm_access_channels=None,
    include_hbm_latency=None,
    hbm_latency_channels=None,
):
    """Run reusable latency analysis for GEMM and non-GEMM kernels.

    Common runtime phases are always reported. GEMM-specific sync/MPM analysis
    is enabled when workload.is_gemm or include_gemm=True.
    """
    workload = workload or Workload()
    paths = paths or cycle_util.DEFAULT_GEMM_PATHS
    fsdb_path = _path_or_default(fsdb_path, paths.fsdb_path)
    simv_log = _path_or_default(simv_log, _default_simv_log(fsdb_path))
    elf_path = _path_or_default(elf_path, cycle_util.DEFAULT_KERNEL_ELF)
    include_gemm = workload.is_gemm if include_gemm is None else include_gemm
    include_intervals = include_gemm if include_intervals is None else include_intervals
    include_hbm_access = include_gemm if include_hbm_access is None else include_hbm_access
    include_hbm_latency = include_gemm if include_hbm_latency is None else include_hbm_latency

    phases = cycle_util.analyze_kernel_agnostic_phases(
        fsdb_path=fsdb_path,
        simv_log=simv_log,
        elf_path=elf_path,
        paths=paths,
        strict=strict,
    )

    sync_wait = None
    mpm_accel = None
    util_summary = None
    fire_intervals = None
    hbm_access_intervals = None
    hbm_read_latency = None
    if include_gemm:
        sync_wait = cycle_util.analyze_sync_wait(fsdb_path, paths)
        mpm_accel = cycle_util.analyze_mpm_accel(fsdb_path, paths, strict=strict)
        util_summary = build_util_summary(phases, mpm_accel)

    if include_intervals:
        fire_intervals = cycle_util.analyze_signal_intervals(
            fsdb_path=fsdb_path,
            paths=paths,
            specs=interval_specs,
            groups=interval_groups,
            kind='fire',
            strict=strict,
        )

    if include_hbm_access:
        hbm_access_intervals = cycle_util.analyze_hbm_access_intervals(
            fsdb_path=fsdb_path,
            paths=paths,
            channels=hbm_access_channels,
            strict=strict,
        )

    if include_hbm_latency:
        hbm_read_latency = cycle_util.analyze_hbm_read_latency(
            fsdb_path=fsdb_path,
            paths=paths,
            channels=hbm_latency_channels,
            strict=strict,
        )

    result = RunResult(
        workload=workload,
        fsdb_path=fsdb_path,
        simv_log=simv_log,
        elf_path=elf_path,
        phases=phases,
        sync_wait=sync_wait,
        mpm_accel=mpm_accel,
        util_summary=util_summary,
        fire_intervals=fire_intervals,
        hbm_access_intervals=hbm_access_intervals,
        hbm_read_latency=hbm_read_latency,
    )

    if show:
        print(f'workload={workload.name} type={workload.kernel_type} m={workload.m} n={workload.n} k={workload.k}')
        print(f'fsdb={fsdb_path}')
        print(f'simv_log={simv_log}')
        print(f'elf={elf_path}')

        phase_cols = [
            'phase', 'start_cycle', 'end_cycle', 'cycles', 'count',
            'dispatch_count', 'commit_count', 'symbols', 'note',
        ]
        display(phases[[col for col in phase_cols if col in phases.columns]])

        if sync_wait is not None:
            display(sync_wait)
        if mpm_accel is not None:
            display(mpm_accel)
        if util_summary is not None:
            display(util_summary)
        if fire_intervals is not None:
            display(fire_intervals)
        if hbm_access_intervals is not None:
            display(hbm_access_intervals)
        if hbm_read_latency is not None:
            display(hbm_read_latency)

    return result


In [4]:
def _parse_fpint_workload(trace_dir):
    trace_dir = Path(trace_dir)
    match = re.search(r'm(\d+)_k(\d+)_n(\d+)', trace_dir.name)
    if not match:
        return Workload(name=trace_dir.name, kernel_type='generic')
    m, k, n = map(int, match.groups())
    return Workload(name=trace_dir.name, kernel_type='gemm', m=m, n=n, k=k)


def _annotate(df, workload):
    if df is None or df.empty:
        return pd.DataFrame()
    out = df.copy()
    out.insert(0, 'trace', workload.name)
    out.insert(1, 'm', workload.m)
    out.insert(2, 'k', workload.k)
    out.insert(3, 'n', workload.n)
    return out


def _compact_phase_view(phase_summary):
    order = [
        'kernel_busy', 'runtime_bootstrap_to_user', 'tag_init', 'warp_spawn', 'user_kernel_body',
        'runtime_exit', 'perf_dump', 'exit_final_to_fence', 'fence_wait',
        'cache_flush_tags', 'host_done_polling',
    ]
    table = phase_summary[phase_summary['phase'].isin(order)].pivot_table(
        index='trace', columns='phase', values='cycles', aggfunc='first'
    )
    return table[[col for col in order if col in table.columns]]


def _compact_sync_view(sync_summary):
    if sync_summary.empty:
        return sync_summary
    order = ['G0', 'G1', 'O', 'W0', 'SZ0', 'W1', 'SZ1', 'T0', 'T1']
    table = sync_summary.pivot_table(
        index='trace', columns='wait_reg_name', values='cycles', aggfunc='first'
    )
    return table[[col for col in order if col in table.columns]]


def _compact_mpm_view(mpm_summary):
    if mpm_summary.empty:
        return mpm_summary

    metrics = [
        ('mxu', 'gemm_total_cycles', 'gemm_total'),
        ('mxu', 'gemm_compute_cycles', 'compute'),
        ('mxu', 'gemm_stall_cycles', 'stall'),
        ('mxu', 'gemm_job_count', 'jobs'),
        ('mxu', 'achieved_flops_per_cycle_total', 'flops_per_cycle'),
        ('hbm_dma', 'rd_bytes', 'hbm_rd_bytes'),
        ('hbm_dma', 'wr_bytes', 'hbm_wr_bytes'),
        ('hbm_dma', 'active_cycles', 'hbm_active'),
        ('hbm_dma', 'bandwidth_bytes_per_active_cycle', 'hbm_bw_active_Bpc'),
        ('hbm_dma', 'bandwidth_bytes_per_busy_cycle', 'hbm_bw_busy_Bpc'),
        ('hbm_dma', 'active_imbalance_pct', 'hbm_active_imbalance_pct'),
        ('ldma_input', 'active_cycles', 'ldma_input_active'),
        ('ldma_weight', 'active_cycles', 'ldma_weight_active'),
        ('ldma_sz', 'active_cycles', 'ldma_sz_active'),
        ('ldma_output', 'active_cycles', 'ldma_output_active'),
    ]

    rows = []
    for trace, sub in mpm_summary.groupby('trace', sort=True):
        row = {'trace': trace}
        for section, metric, name in metrics:
            value = sub.loc[(sub['section'] == section) & (sub['metric'] == metric), 'value']
            row[name] = None if value.empty else value.iloc[0]
        rows.append(row)
    return pd.DataFrame(rows).set_index('trace')


def _compact_util_view(util_summary):
    if util_summary.empty:
        return util_summary

    metrics = [
        ('mxu', 'active_pct_user_kernel_body', 'mxu_active_user_pct'),
        ('mxu', 'compute_pct_gemm_total', 'mxu_compute_pct'),
        ('mxu', 'stall_pct_gemm_total', 'mxu_stall_pct'),
        ('mxu', 'flops_per_gemm_cycle', 'flops_per_cycle'),
        ('mxu_port', 'input_util_pct_compute', 'mxu_input_util_compute'),
        ('mxu_port', 'weight_util_pct_compute', 'mxu_weight_util_compute'),
        ('mxu_port', 'output_util_pct_compute', 'mxu_output_util_compute'),
        ('hbm_dma', 'active_pct_user_kernel_body', 'hbm_active_user_pct'),
        ('hbm_dma', 'bandwidth_bytes_per_active_cycle', 'hbm_bw_active_Bpc'),
        ('ldma_input', 'active_pct_gemm_total', 'ldma_input_active_gemm_pct'),
        ('ldma_weight', 'active_pct_gemm_total', 'ldma_weight_active_gemm_pct'),
        ('ldma_sz', 'active_pct_gemm_total', 'ldma_sz_active_gemm_pct'),
        ('ldma_output', 'active_pct_gemm_total', 'ldma_output_active_gemm_pct'),
    ]

    rows = []
    for trace, sub in util_summary.groupby('trace', sort=True):
        row = {'trace': trace}
        for section, metric, name in metrics:
            value = sub.loc[(sub['section'] == section) & (sub['metric'] == metric), 'value']
            row[name] = None if value.empty else value.iloc[0]
        rows.append(row)
    return pd.DataFrame(rows).set_index('trace')


def _compact_fire_interval_view(fire_intervals):
    if fire_intervals.empty:
        return fire_intervals

    preferred = [
        ('mxu', 'input'),
        ('mxu', 'weight'),
        ('mxu', 'psum'),
        ('mxu', 'output'),
        ('ldma_input', 'src_rd_req'),
        ('ldma_input', 'dst_wr'),
        ('ldma_weight', 'src_rd_req'),
        ('ldma_weight', 'dst_wr'),
        ('ldma_sz', 'src_rd_req'),
        ('ldma_sz', 'dst_wr'),
        ('ldma_output', 'src_rd_req'),
        ('ldma_output', 'dst_wr'),
    ]

    rows = []
    for trace, sub in fire_intervals.groupby('trace', sort=True):
        row = {'trace': trace}
        for section, stream in preferred:
            match = sub[(sub['section'] == section) & (sub['stream'] == stream)]
            prefix = f'{section}_{stream}'
            if match.empty:
                row[f'{prefix}_p50'] = None
                row[f'{prefix}_p90'] = None
                row[f'{prefix}_max_burst'] = None
                row[f'{prefix}_consec_pct'] = None
            else:
                item = match.iloc[0]
                row[f'{prefix}_p50'] = item['p50_interval']
                row[f'{prefix}_p90'] = item['p90_interval']
                row[f'{prefix}_max_burst'] = item['max_burst_len']
                row[f'{prefix}_consec_pct'] = item['consecutive_interval_pct']
        rows.append(row)
    return pd.DataFrame(rows).set_index('trace')


def _compact_hbm_access_interval_view(hbm_access_intervals):
    if hbm_access_intervals.empty:
        return hbm_access_intervals

    rows = []
    for trace, sub in hbm_access_intervals.groupby('trace', sort=True):
        row = {'trace': trace}
        for stream in ('read_req', 'read_rsp', 'write_req'):
            match = sub[sub['stream'] == stream]
            prefix = f'hbm_{stream}'
            if match.empty:
                row[f'{prefix}_events'] = None
                row[f'{prefix}_p50_min'] = None
                row[f'{prefix}_p90_max'] = None
                row[f'{prefix}_max_burst'] = None
                row[f'{prefix}_consec_pct_mean'] = None
            else:
                row[f'{prefix}_events'] = match['event_count'].sum()
                row[f'{prefix}_p50_min'] = match['p50_interval'].min()
                row[f'{prefix}_p90_max'] = match['p90_interval'].max()
                row[f'{prefix}_max_burst'] = match['max_burst_len'].max()
                row[f'{prefix}_consec_pct_mean'] = match['consecutive_interval_pct'].mean()
        rows.append(row)
    return pd.DataFrame(rows).set_index('trace')


def _compact_hbm_read_latency_view(hbm_read_latency):
    if hbm_read_latency.empty:
        return hbm_read_latency

    metrics = [
        ('req_count', 'hbm_read_req_count'),
        ('rsp_count', 'hbm_read_rsp_count'),
        ('matched_count', 'hbm_read_matched_count'),
        ('mean_latency', 'hbm_read_mean_latency'),
        ('p50_latency', 'hbm_read_p50_latency'),
        ('p90_latency', 'hbm_read_p90_latency'),
        ('p99_latency', 'hbm_read_p99_latency'),
        ('max_latency', 'hbm_read_max_latency'),
        ('unmatched_req_count', 'hbm_read_unmatched_req_count'),
        ('orphan_rsp_count', 'hbm_read_orphan_rsp_count'),
    ]

    rows = []
    for trace, sub in hbm_read_latency.groupby('trace', sort=True):
        aggregate = sub[sub['section'] == 'hbm_dma_all']
        item = aggregate.iloc[0] if not aggregate.empty else sub.iloc[0]
        row = {'trace': trace}
        for source, name in metrics:
            row[name] = item.get(source)
        rows.append(row)
    return pd.DataFrame(rows).set_index('trace')


def _has_glob(value):
    return any(ch in str(value) for ch in '*?[')


def _resolve_trace_dirs(log_root, pattern, trace_dirs=None):
    if trace_dirs is None:
        traces = sorted(path for path in log_root.glob(pattern) if path.is_dir())
        if not traces:
            raise FileNotFoundError(f'No traces matched {log_root / pattern}')
        return traces

    if isinstance(trace_dirs, (str, Path)):
        trace_dirs = [trace_dirs]

    traces = []
    missing = []
    for item in trace_dirs:
        text = str(item)
        raw_path = Path(text).expanduser()
        search_paths = [raw_path] if raw_path.is_absolute() else [log_root / raw_path, REPO_ROOT / raw_path]
        if _has_glob(text):
            matches = []
            for pattern_path in search_paths:
                matches.extend(Path(path) for path in glob.glob(str(pattern_path)) if Path(path).is_dir())
            matches = sorted(matches)
            if not matches:
                missing.append(' or '.join(str(path) for path in search_paths))
            traces.extend(matches)
        else:
            match = next((path for path in search_paths if path.is_dir()), None)
            if match is not None:
                traces.append(match)
            else:
                missing.append(' or '.join(str(path) for path in search_paths))

    unique = []
    seen = set()
    for trace_dir in traces:
        resolved = trace_dir.resolve()
        if resolved not in seen:
            seen.add(resolved)
            unique.append(resolved)

    if missing:
        raise FileNotFoundError('No trace directory matched: ' + ', '.join(missing))
    if not unique:
        raise FileNotFoundError('No trace directories selected')
    return unique


def run_many_fpint_improve(
    log_root=None,
    pattern='fpint_improve_*',
    trace_dirs=None,
    *,
    elf_path=None,
    paths=None,
    include_gemm=True,
    write_csv=True,
    output_dir=None,
    show=True,
    include_intervals=True,
    interval_groups=None,
    interval_specs=None,
    include_hbm_access=True,
    hbm_access_channels=None,
    include_hbm_latency=True,
    hbm_latency_channels=None,
):
    log_root = _path_or_default(log_root, REPO_ROOT / 'build' / 'logs')
    output_dir = _path_or_default(output_dir, LATENCY_DIR)
    elf_path = _path_or_default(elf_path, cycle_util.DEFAULT_KERNEL_ELF)
    paths = paths or cycle_util.DEFAULT_GEMM_PATHS

    traces = _resolve_trace_dirs(log_root, pattern, trace_dirs)

    phase_rows = []
    sync_rows = []
    mpm_rows = []
    util_rows = []
    interval_rows = []
    hbm_access_interval_rows = []
    hbm_read_latency_rows = []
    results = {}

    for trace_dir in traces:
        workload = _parse_fpint_workload(trace_dir)
        fsdb_path = trace_dir / 'xrtsim_vcs' / 'vcs_cosim.fsdb'
        simv_log = trace_dir / 'xrtsim_vcs' / 'simv.log'
        if not fsdb_path.exists() or not simv_log.exists():
            print(f'skip missing trace files: {trace_dir}')
            continue

        result = run(
            fsdb_path=fsdb_path,
            workload=workload,
            simv_log=simv_log,
            elf_path=elf_path,
            paths=paths,
            include_gemm=include_gemm and workload.is_gemm,
            show=False,
            include_intervals=include_intervals and workload.is_gemm,
            interval_groups=interval_groups,
            interval_specs=interval_specs,
            include_hbm_access=include_hbm_access and workload.is_gemm,
            hbm_access_channels=hbm_access_channels,
            include_hbm_latency=include_hbm_latency and workload.is_gemm,
            hbm_latency_channels=hbm_latency_channels,
        )
        results[workload.name] = result
        phase_rows.append(_annotate(result.phases, workload))
        sync_rows.append(_annotate(result.sync_wait, workload))
        mpm_rows.append(_annotate(result.mpm_accel, workload))
        util_rows.append(_annotate(result.util_summary, workload))
        interval_rows.append(_annotate(result.fire_intervals, workload))
        hbm_access_interval_rows.append(_annotate(result.hbm_access_intervals, workload))
        hbm_read_latency_rows.append(_annotate(result.hbm_read_latency, workload))

    phase_summary = pd.concat(phase_rows, ignore_index=True) if phase_rows else pd.DataFrame()
    sync_summary = pd.concat(sync_rows, ignore_index=True) if sync_rows else pd.DataFrame()
    mpm_summary = pd.concat(mpm_rows, ignore_index=True) if mpm_rows else pd.DataFrame()
    util_summary = pd.concat(util_rows, ignore_index=True) if util_rows else pd.DataFrame()
    fire_interval_summary = pd.concat(interval_rows, ignore_index=True) if interval_rows else pd.DataFrame()
    hbm_access_interval_summary = pd.concat(hbm_access_interval_rows, ignore_index=True) if hbm_access_interval_rows else pd.DataFrame()
    hbm_read_latency_summary = pd.concat(hbm_read_latency_rows, ignore_index=True) if hbm_read_latency_rows else pd.DataFrame()

    phase_compact = _compact_phase_view(phase_summary) if not phase_summary.empty else pd.DataFrame()
    sync_compact = _compact_sync_view(sync_summary) if not sync_summary.empty else pd.DataFrame()
    mpm_compact = _compact_mpm_view(mpm_summary) if not mpm_summary.empty else pd.DataFrame()
    util_compact = _compact_util_view(util_summary) if not util_summary.empty else pd.DataFrame()
    fire_interval_compact = (
        _compact_fire_interval_view(fire_interval_summary)
        if not fire_interval_summary.empty else pd.DataFrame()
    )
    hbm_access_interval_compact = (
        _compact_hbm_access_interval_view(hbm_access_interval_summary)
        if not hbm_access_interval_summary.empty else pd.DataFrame()
    )
    hbm_read_latency_compact = (
        _compact_hbm_read_latency_view(hbm_read_latency_summary)
        if not hbm_read_latency_summary.empty else pd.DataFrame()
    )
    bottleneck_summary = cycle_util.classify_bottlenecks(
        phase_compact,
        mpm_compact,
        util_compact,
        sync_compact,
    )

    if write_csv:
        output_dir.mkdir(parents=True, exist_ok=True)
        phase_summary.to_csv(output_dir / 'fpint_improve_phase_summary.csv', index=False)
        sync_summary.to_csv(output_dir / 'fpint_improve_sync_wait_summary.csv', index=False)
        mpm_summary.to_csv(output_dir / 'fpint_improve_mpm_summary.csv', index=False)
        util_summary.to_csv(output_dir / 'fpint_improve_util_summary.csv', index=False)
        fire_interval_summary.to_csv(output_dir / 'fpint_improve_fire_interval_summary.csv', index=False)
        hbm_access_interval_summary.to_csv(output_dir / 'fpint_improve_hbm_access_interval_summary.csv', index=False)
        hbm_read_latency_summary.to_csv(output_dir / 'fpint_improve_hbm_read_latency_summary.csv', index=False)

        phase_compact.to_csv(output_dir / 'fpint_improve_phase_compact.csv')
        sync_compact.to_csv(output_dir / 'fpint_improve_sync_wait_compact.csv')
        mpm_compact.to_csv(output_dir / 'fpint_improve_mpm_compact.csv')
        util_compact.to_csv(output_dir / 'fpint_improve_util_compact.csv')
        fire_interval_compact.to_csv(output_dir / 'fpint_improve_fire_interval_compact.csv')
        hbm_access_interval_compact.to_csv(output_dir / 'fpint_improve_hbm_access_interval_compact.csv')
        hbm_read_latency_compact.to_csv(output_dir / 'fpint_improve_hbm_read_latency_compact.csv')
        bottleneck_summary.to_csv(output_dir / 'fpint_improve_bottleneck_summary.csv', index=False)

    if show:
        print(f'traces={len(results)} log_root={log_root}')
        if trace_dirs is not None:
            print('selected_traces=' + ', '.join(path.name for path in traces))
        if not phase_compact.empty:
            display(phase_compact)
        if not sync_compact.empty:
            display(sync_compact)
        if not mpm_compact.empty:
            display(mpm_compact)
        if not util_compact.empty:
            display(util_compact)
        if not fire_interval_compact.empty:
            display(fire_interval_compact)
        if not hbm_access_interval_compact.empty:
            display(hbm_access_interval_compact)
        if not hbm_read_latency_compact.empty:
            display(hbm_read_latency_compact)
        if not bottleneck_summary.empty:
            display(bottleneck_summary)

    return {
        'results': results,
        'phase_summary': phase_summary,
        'sync_summary': sync_summary,
        'mpm_summary': mpm_summary,
        'util_summary': util_summary,
        'fire_interval_summary': fire_interval_summary,
        'hbm_access_interval_summary': hbm_access_interval_summary,
        'hbm_read_latency_summary': hbm_read_latency_summary,
        'phase_compact': phase_compact,
        'sync_compact': sync_compact,
        'mpm_compact': mpm_compact,
        'util_compact': util_compact,
        'fire_interval_compact': fire_interval_compact,
        'hbm_access_interval_compact': hbm_access_interval_compact,
        'hbm_read_latency_compact': hbm_read_latency_compact,
        'bottleneck_summary': bottleneck_summary,
    }


In [5]:
SELECTED_TRACE_DIRS = [
    'fpint_improve_m1_k128_n64_tcol1',
    'fpint_improve_m1_k128_n64_tcol32',
]

fpint_improve_results = run_many_fpint_improve(
    trace_dirs=SELECTED_TRACE_DIRS,
    output_dir=LATENCY_DIR / 'fpint_improve_m1_k128_n64_tcol_compare',
)

FSDB: /home/jaeyongjang/project.local/vortex/build/logs/fpint_improve_m1_k128_n64_tcol1/xrtsim_vcs/vcs_cosim.fsdb
simv.log: /home/jaeyongjang/project.local/vortex/build/logs/fpint_improve_m1_k128_n64_tcol1/xrtsim_vcs/simv.log
ELF: /home/jaeyongjang/project.local/vortex/build/tests/regression/fpint_gemm_ffn_hw/kernel.elf
Window: bt=None, et=None, clock_period_ps=10000
FSDB: /home/jaeyongjang/project.local/vortex/build/logs/fpint_improve_m1_k128_n64_tcol1/xrtsim_vcs/vcs_cosim.fsdb
Window: bt=None, et=None, time_unit=1ps
total_wait_active_cycles=4201
FSDB: /home/jaeyongjang/project.local/vortex/build/logs/fpint_improve_m1_k128_n64_tcol1/xrtsim_vcs/vcs_cosim.fsdb
Window: bt=None, et=None
FSDB: /home/jaeyongjang/project.local/vortex/build/logs/fpint_improve_m1_k128_n64_tcol1/xrtsim_vcs/vcs_cosim.fsdb
Window: bt=None, et=None, clock_period_ps=10000, kind=fire
burst_gap_cycles=1, sample_on_clk=False
FSDB: /home/jaeyongjang/project.local/vortex/build/logs/fpint_improve_m1_k128_n64_tcol1/xrtsim

phase,runtime_bootstrap_to_user,tag_init,warp_spawn,user_kernel_body,runtime_exit,perf_dump,exit_final_to_fence,fence_wait,cache_flush_tags,host_done_polling
trace,,,,,,,,,,
fpint_improve_m1_k128_n64_tcol1,4588,2048,266,6708,538,547,590,2078,2048,19
fpint_improve_m1_k128_n64_tcol32,4588,2048,266,4688,538,547,590,2078,2048,7


wait_reg_name,G0,G1,O,W0,SZ0,W1,SZ1,T0,T1
trace,,,,,,,,,
fpint_improve_m1_k128_n64_tcol1,1848,1856,269,105,55,32,32,2,2
fpint_improve_m1_k128_n64_tcol32,856,864,269,105,55,32,32,2,2


,gemm_total,compute,stall,jobs,flops_per_cycle,hbm_rd_bytes,hbm_wr_bytes,hbm_active,hbm_bw_active_Bpc,hbm_bw_busy_Bpc,hbm_active_imbalance_pct,ldma_input_active,ldma_weight_active,ldma_sz_active,ldma_output_active
trace,,,,,,,,,,,,,,,
fpint_improve_m1_k128_n64_tcol1,4822.0,3384.0,64.0,64.0,3.39776,41984.0,512.0,2539.0,100.226415,2.927125,33.962264,640.0,1089.0,1280.0,80.0
fpint_improve_m1_k128_n64_tcol32,2838.0,1400.0,64.0,64.0,5.77308,41984.0,512.0,2544.0,99.755869,3.400224,34.037559,640.0,1089.0,1280.0,80.0


,mxu_active_user_pct,mxu_compute_pct,mxu_stall_pct,flops_per_cycle,mxu_input_util_compute,mxu_weight_util_compute,mxu_output_util_compute,hbm_active_user_pct,hbm_bw_active_Bpc,ldma_input_active_gemm_pct,ldma_weight_active_gemm_pct,ldma_sz_active_gemm_pct,ldma_output_active_gemm_pct
trace,,,,,,,,,,,,,
fpint_improve_m1_k128_n64_tcol1,71.884317,70.178349,1.327250,3.39776,1.891253,15.130024,0.236407,37.850328,100.226415,13.272501,22.583990,26.545002,1.659063
fpint_improve_m1_k128_n64_tcol32,60.537543,49.330514,2.255109,5.77308,4.571429,36.571429,0.571429,54.266212,99.755869,22.551092,38.372093,45.102185,2.818887


,mxu_input_p50,mxu_input_p90,mxu_input_max_burst,mxu_input_consec_pct,mxu_weight_p50,mxu_weight_p90,mxu_weight_max_burst,mxu_weight_consec_pct,mxu_psum_p50,mxu_psum_p90,...,ldma_sz_dst_wr_max_burst,ldma_sz_dst_wr_consec_pct,ldma_output_src_rd_req_p50,ldma_output_src_rd_req_p90,ldma_output_src_rd_req_max_burst,ldma_output_src_rd_req_consec_pct,ldma_output_dst_wr_p50,ldma_output_dst_wr_p90,ldma_output_dst_wr_max_burst,ldma_output_dst_wr_consec_pct
trace,,,,,,,,,,,,,,,,,,,,,
fpint_improve_m1_k128_n64_tcol1,67.0,67.0,1,0.0,1.0,59.0,8,87.475538,67.0,133.0,...,1,0.0,40.0,917.2,1,0.0,40.0,917.2,1,0.0
fpint_improve_m1_k128_n64_tcol32,36.0,36.0,1,0.0,1.0,28.0,8,87.475538,36.0,71.0,...,1,0.0,40.0,520.4,1,0.0,40.0,520.4,1,0.0


,hbm_read_req_events,hbm_read_req_p50_min,hbm_read_req_p90_max,hbm_read_req_max_burst,hbm_read_req_consec_pct_mean,hbm_read_rsp_events,hbm_read_rsp_p50_min,hbm_read_rsp_p90_max,hbm_read_rsp_max_burst,hbm_read_rsp_consec_pct_mean,hbm_write_req_events,hbm_write_req_p50_min,hbm_write_req_p90_max,hbm_write_req_max_burst,hbm_write_req_consec_pct_mean
trace,,,,,,,,,,,,,,,
fpint_improve_m1_k128_n64_tcol1,656,1.0,18.0,2,48.350999,656,2.0,19.0,2,48.320497,8,0.0,917.2,1,0.0
fpint_improve_m1_k128_n64_tcol32,656,1.0,19.0,2,48.493976,656,1.0,19.0,2,48.779930,8,0.0,520.4,1,0.0


,hbm_read_req_count,hbm_read_rsp_count,hbm_read_matched_count,hbm_read_mean_latency,hbm_read_p50_latency,hbm_read_p90_latency,hbm_read_p99_latency,hbm_read_max_latency,hbm_read_unmatched_req_count,hbm_read_orphan_rsp_count
trace,,,,,,,,,,
fpint_improve_m1_k128_n64_tcol1,656,656,656,2.274390,3.0,3.0,5.0,5,0,0
fpint_improve_m1_k128_n64_tcol32,656,656,656,2.280488,3.0,3.0,5.0,6,0,0
